Instalación de dependencias

*   LangChain → framework principal del TP.
*   Embeddings → convertir nuestros textos en vectores.
*   Chroma → almacenar y buscar esos vectores.
*   OpenAI → modelo de lenguaje.

In [1]:
!pip install -q langchain langchain-community langchain-openai chromadb python-dotenv
!pip install -q unstructured

Conexión a Drive

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
#pruebas
import os

PROJECT_PATH = "/content/drive/MyDrive/TravelAI"
RAG_PATH = f"{PROJECT_PATH}/rag"

print(os.listdir(PROJECT_PATH))
print(os.listdir(RAG_PATH))

['rag', 'src', 'notebooks']
['destinos', 'intereses', 'planificacion']


In [4]:
import os

for root, dirs, files in os.walk(RAG_PATH):
    for file in files:
        if file.endswith(".md"):
            print(os.path.join(root, file))

/content/drive/MyDrive/TravelAI/rag/destinos/buenosAires.md
/content/drive/MyDrive/TravelAI/rag/destinos/cordoba.md
/content/drive/MyDrive/TravelAI/rag/destinos/rosario.md
/content/drive/MyDrive/TravelAI/rag/intereses/gastronomia.md
/content/drive/MyDrive/TravelAI/rag/intereses/naturaleza.md
/content/drive/MyDrive/TravelAI/rag/intereses/historia.md
/content/drive/MyDrive/TravelAI/rag/intereses/cultura.md
/content/drive/MyDrive/TravelAI/rag/planificacion/presupuesto.md
/content/drive/MyDrive/TravelAI/rag/planificacion/restricciones.md
/content/drive/MyDrive/TravelAI/rag/planificacion/reglasItinerario.md


Cargar los documentos con LangChain

In [5]:
from langchain_community.document_loaders import DirectoryLoader

loader = DirectoryLoader(
    RAG_PATH,
    glob="**/*.md"
)

documents = loader.load()

print(f"Documentos cargados: {len(documents)}")

/tmp/ipykernel_15644/1970481706.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader


Documentos cargados: 10


Dividir los documentos en chunks

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"Chunks generados: {len(chunks)}")

Chunks generados: 169


In [7]:
!pip install -U google-genai

In [8]:
import os
from google.colab import userdata
from google import genai

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

## PRUEBA

texto_prueba = "Quiero conocer la historia de Córdoba"

resultado = client.models.embed_content(
    model="gemini-embedding-001",
    contents=texto_prueba
)

vector = resultado.embeddings[0].values

print("Cantidad de valores:", len(vector))
print("Primeros valores:", vector[:10])

Cantidad de valores: 3072
Primeros valores: [-0.010480844, -0.025091834, 0.044006113, -0.056719024, 0.0039966796, 0.0013464583, -0.015473723, 0.010241068, -0.002296018, 0.008520809]


Adaptador Gemini

In [9]:
!pip install -U chromadb langchain-community
from langchain_core.embeddings import Embeddings
class GeminiEmbeddings(Embeddings):

    def embed_documents(self, texts):
        resultado = client.models.embed_content(
            model="gemini-embedding-001",
            contents=texts
        )

        return [embedding.values for embedding in resultado.embeddings]

    def embed_query(self, text):
        resultado = client.models.embed_content(
            model="gemini-embedding-001",
            contents=text
        )

        return resultado.embeddings[0].values

gemini_embeddings = GeminiEmbeddings()

texto_prueba = "Quiero conocer la historia de Córdoba"

vector = gemini_embeddings.embed_query(texto_prueba)

print("Cantidad de valores:", len(vector))
print("Primeros valores:", vector[:10])

Cantidad de valores: 3072
Primeros valores: [-0.010480844, -0.025091834, 0.044006113, -0.056719024, 0.0039966796, 0.0013464583, -0.015473723, 0.010241068, -0.002296018, 0.008520809]


Crear chroma

In [10]:
import os
import shutil
import time
from langchain_community.vectorstores import Chroma

persist_path = f"{PROJECT_PATH}/vectorstore"

# 1. Si la carpeta ya existe, la borramos para no duplicar los 80 chunks que ya se guardaron
if os.path.exists(persist_path):
    shutil.rmtree(persist_path)

# 2. Inicializamos Chroma limpio
vectorstore = Chroma(
    embedding_function=gemini_embeddings,
    persist_directory=persist_path
)

BATCH_SIZE = 70
print(f"Chunks totales a guardar: {len(chunks)}")

# 3. Guardamos en lotes con pausa de seguridad
for i in range(0, len(chunks), BATCH_SIZE):
    lote = chunks[i:i + BATCH_SIZE]
    vectorstore.add_documents(documents=lote)
    print(f"Guardados {min(i + BATCH_SIZE, len(chunks))} de {len(chunks)} chunks...")

    # Si todavía faltan chunks por guardar, esperamos para no superar el límite por minuto
    if i + BATCH_SIZE < len(chunks):
        print("Esperando 35 segundos para respetar la cuota gratuita de Google...")
        time.sleep(35)

print("\n¡Completado con éxito!")
print("Cantidad de chunks almacenados:", vectorstore._collection.count())
print(os.listdir(PROJECT_PATH))

/tmp/ipykernel_15644/3608920832.py:13: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


Chunks totales a guardar: 169
Guardados 70 de 169 chunks...
Esperando 35 segundos para respetar la cuota gratuita de Google...
Guardados 140 de 169 chunks...
Esperando 35 segundos para respetar la cuota gratuita de Google...
Guardados 169 de 169 chunks...

¡Completado con éxito!
Cantidad de chunks almacenados: 169
['rag', 'src', 'notebooks', 'vectorstore']


Crear el Retriever

In [11]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

consulta = "Quiero hacer actividades relacionadas con la naturaleza en Córdoba"

resultados = retriever.invoke(consulta)

for i, doc in enumerate(resultados):
    print(f"\n--- Resultado {i+1} ---")
    print(doc.page_content[:1000])

## consulta 1

consulta = "Quiero conocer lugares históricos de Buenos Aires"

resultados = retriever.invoke(consulta)

for i, doc in enumerate(resultados):
    print(f"\n--- Resultado {i+1} ---")
    print(doc.page_content[:700])

## consulta 2

consulta = "Busco actividades al aire libre y contacto con la naturaleza"

resultados = retriever.invoke(consulta)

for i, doc in enumerate(resultados):
    print(f"\n--- Resultado {i+1} ---")
    print(doc.page_content[:700])

## consulta 3

consulta = "Tengo poco presupuesto y quiero organizar un viaje económico"

resultados = retriever.invoke(consulta)

for i, doc in enumerate(resultados):
    print(f"\n--- Resultado {i+1} ---")
    print(doc.page_content[:700])


--- Resultado 1 ---
Dentro de la ciudad pueden considerarse:

Parque Sarmiento

Espacios verdes de Ciudad Universitaria

Paseos urbanos

Plazas y parques

Sectores recreativos

Las actividades relacionadas con naturaleza pueden incluir:

Caminatas

Ciclismo

Paseos

Actividades deportivas

Observación del paisaje

Fotografía

Actividades recreativas

Si el usuario desea realizar excursiones fuera de la ciudad, el sistema debe diferenciar entre actividades urbanas y excursiones hacia localidades o áreas naturales de la provincia.

La elección de una actividad debe considerar la época del año, las condiciones climáticas, el tiempo disponible y la disponibilidad actual de la actividad.

Sierras y excursiones desde Córdoba

La ciudad de Córdoba puede funcionar como punto de partida para visitar diferentes localidades y áreas naturales de la provincia.

Entre los destinos serranos y turísticos que pueden considerarse se encuentran:

Villa Carlos Paz

La Cumbrecita

Alta Gracia

La Falda

-

Conectar LLM

In [14]:
!pip install -U google-genai

In [15]:
import os
from google.colab import userdata
from google import genai

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

respuesta = client.interactions.create(
    model="gemini-3.6-flash",
    input="Explicame brevemente qué es el turismo cultural."
)

print(respuesta.output_text)

El **turismo cultural** es un tipo de viaje en el que la motivación principal del turista es **conocer, aprender y experimentar la cultura de un destino**.

A diferencia del turismo de descanso (como el de sol y playa), el viajero cultural busca conectarse con la identidad de un lugar a través de actividades como:

*   **Patrimonio histórico:** Visitar monumentos, ruinas, castillos o sitios arqueológicos.
*   **Arte y museos:** Asistir a galerías, museos, teatros o conciertos.
*   **Tradiciones y festividades:** Participar en fiestas locales, festivales, música y folclore.
*   **Gastronomía:** Probar la comida típica y aprender sobre sus tradiciones culinarias.
*   **Estilo de vida:** Interactuar con la comunidad local para entender sus costumbres diarias.

**En resumen:** Es viajar movido por la curiosidad de descubrir la historia, las tradiciones y la creatividad humana de un lugar, lo que además ayuda a preservar ese patrimonio local.


##Crear el RAG completo

In [16]:
consulta = "Quiero conocer lugares históricos de Buenos Aires"

resultados = retriever.invoke(consulta)

for i, doc in enumerate(resultados):
    print(f"\n--- Resultado {i+1} ---")
    print(doc.page_content[:1000])

#Esto recupera los 4 chunks que Chroma considera más relevantes.


--- Resultado 1 ---
Entre los principales atractivos se encuentran:

Plaza de Mayo Casa Rosada Cabildo Catedral Metropolitana Palacio de Gobierno de la Ciudad Manzana de las Luces Avenida de Mayo Plaza Dorrego Parque Lezama

Esta zona es especialmente adecuada para viajeros interesados en:

Historia Arquitectura Patrimonio Cultura Fotografía Caminatas urbanas

El área puede organizarse como un recorrido histórico debido a la proximidad entre varios de sus principales atractivos.

Plaza de Mayo y entorno La Plaza de Mayo es uno de los espacios históricos y políticos más importantes de Buenos Aires.

Su entorno concentra algunos de los edificios más representativos de la ciudad:

Casa Rosada Cabildo Catedral Metropolitana Palacio de Gobierno de la Ciudad

La zona es especialmente relevante para itinerarios relacionados con:

Historia argentina Arquitectura Patrimonio Política e historia nacional Turismo urbano

Puede combinarse con Avenida de Mayo, el Obelisco, San Telmo y otros puntos 

Crear el contexto

In [17]:
contexto = "\n\n".join(
    doc.page_content
    for doc in resultados
)

print(contexto)

Entre los principales atractivos se encuentran:

Plaza de Mayo Casa Rosada Cabildo Catedral Metropolitana Palacio de Gobierno de la Ciudad Manzana de las Luces Avenida de Mayo Plaza Dorrego Parque Lezama

Esta zona es especialmente adecuada para viajeros interesados en:

Historia Arquitectura Patrimonio Cultura Fotografía Caminatas urbanas

El área puede organizarse como un recorrido histórico debido a la proximidad entre varios de sus principales atractivos.

Plaza de Mayo y entorno La Plaza de Mayo es uno de los espacios históricos y políticos más importantes de Buenos Aires.

Su entorno concentra algunos de los edificios más representativos de la ciudad:

Casa Rosada Cabildo Catedral Metropolitana Palacio de Gobierno de la Ciudad

La zona es especialmente relevante para itinerarios relacionados con:

Historia argentina Arquitectura Patrimonio Política e historia nacional Turismo urbano

Puede combinarse con Avenida de Mayo, el Obelisco, San Telmo y otros puntos del centro.

Historia

Crear el prompt RAG

In [19]:
prompt = f"""
Eres un asistente especializado en planificación de viajes.

Responde la consulta del usuario utilizando únicamente
la información proporcionada en el contexto.

Si la información necesaria no aparece en el contexto,
indica que no dispones de esa información.

CONTEXTO:
{contexto}

CONSULTA DEL USUARIO:
{consulta}

RESPUESTA:
"""

respuesta = client.interactions.create(
    model="gemini-3.6-flash",
    input=prompt
)

print(respuesta.output_text)

Según la información proporcionada en el contexto, si estás interesado en la **historia** de Buenos Aires, se recomienda priorizar y visitar los siguientes lugares y zonas:

* **Plaza de Mayo y su entorno:** Casa Rosada, Cabildo, Catedral Metropolitana y el Palacio de Gobierno de la Ciudad.
* **Avenida de Mayo:** Conecta históricamente la Plaza de Mayo con el área del Congreso.
* **Manzana de las Luces**
* **San Telmo** (incluye la Plaza Dorrego)
* **Parque Lezama**
* **Recoleta**
* **Abasto y Balvanera**
* **Edificios históricos** 

Estas zonas y atractivos permiten organizar recorridos históricos debido a su proximidad e importancia patrimonial y cultural.


## Convertir el RAG en TravelAI

In [20]:
def travel_ai(consulta):

    # 1. Recuperar información relevante
    resultados = retriever.invoke(consulta)

    # 2. Construir contexto
    contexto = "\n\n".join(
        doc.page_content
        for doc in resultados
    )

    # 3. Construir prompt
    prompt = f"""
Eres TravelAI, un sistema inteligente de planificación
de viajes.

Tu objetivo es generar recomendaciones e itinerarios
personalizados utilizando la información recuperada
desde la base de conocimiento.

Debes:

- utilizar la información del contexto;
- respetar las preferencias del usuario;
- respetar las restricciones indicadas;
- no inventar información que no esté respaldada
  por el contexto;
- indicar cuando falta información;
- organizar la respuesta de forma clara.

CONTEXTO RECUPERADO:
{contexto}

SOLICITUD DEL USUARIO:
{consulta}

RESPUESTA DE TRAVELAI:
"""

    # 4. Generar respuesta
    respuesta = client.interactions.create(
        model="gemini-3.6-flash",
        input=prompt
    )

    return respuesta.output_text

In [21]:
consulta = """
Quiero viajar 5 días a Buenos Aires.
Somos dos personas.
Tenemos un presupuesto medio.
Nos interesa la gastronomía y la cultura.
Preferimos caminar antes que utilizar transporte.
No queremos hacer más de 3 actividades por día.
"""

respuesta = travel_ai(consulta)

print(respuesta)

¡Hola! Soy **TravelAI**, tu sistema inteligente de planificación de viajes. 

A continuación, presento una propuesta de itinerario personalizada de **5 días para 2 personas**, optimizada para disfrutar a pie concentrando las actividades por zonas/barrios, respetando el límite de **máximo 3 actividades por día** y enfocada en **gastronomía y cultura**.

---

### 🗺️ Itinerario Propuesto: Buenos Aires a Pie (Cultura y Gastronomía)

#### **Día 1: Casco Histórico y San Nicolás (Historia y Arquitectura)**
*Diseñado para recorrer caminando la cuna histórica y arquitectónica de la ciudad.*
1. **Recorrido cultural e histórico:** Caminata a pie por Plaza de Mayo, Casa Rosada, Cabildo y Catedral Metropolitana.
2. **Arquitectura y Teatro:** Recorrido por la Avenida de Mayo hasta el Teatro Colón y la Manzana de las Luces.
3. **Experiencia gastronómica:** Parada en un **Bar Notable** de la zona para disfrutar de un tradicional café o chocolate con churros, o degustar pizza en una pizzería histórica.

## Aplicamos las reglas de TravelAI

In [22]:
def travel_ai(consulta):

    # 1. Recuperar documentos
    resultados = retriever.invoke(consulta)

    # 2. Construir contexto
    contexto = "\n\n".join(
        doc.page_content
        for doc in resultados
    )

    # 3. Prompt
    prompt = f"""
Eres TravelAI, un sistema experto de planificación
inteligente de viajes.

Tu objetivo es generar itinerarios personalizados.

Debes utilizar la información recuperada y aplicar
las reglas de planificación del sistema.

REGLAS IMPORTANTES:

1. Respetar las restricciones indicadas por el usuario.
2. No superar el número máximo de actividades por día
   indicado por el usuario.
3. Tener en cuenta el presupuesto.
4. Priorizar los intereses indicados.
5. Respetar las preferencias de transporte.
6. No inventar información que no esté respaldada
   por el contexto.
7. Si falta información, indicarlo explícitamente.

CONTEXTO RECUPERADO:
{contexto}

SOLICITUD DEL USUARIO:
{consulta}

Genera una respuesta clara y organizada.

Si el usuario solicita un itinerario, organiza la
respuesta por días.

RESPUESTA:
"""

    respuesta = client.interactions.create(
        model="gemini-3.6-flash",
        input=prompt
    )

    return respuesta.output_text

In [23]:
consulta = """
Quiero viajar 5 días a Buenos Aires.

Somos dos personas.
Tenemos presupuesto medio.
Nos interesa la gastronomía y la cultura.
Preferimos caminar antes que usar transporte.
No queremos hacer más de 3 actividades por día.

Generá un itinerario.
"""

print(travel_ai(consulta))

¡Hola! Soy **TravelAI**, tu sistema experto de planificación inteligente de viajes. 

A continuación, presento un itinerario personalizado de **5 días para 2 personas** con presupuesto medio, enfocado en **gastronomía y cultura**, estructurado por zonas cercanas para facilitar los recorridos a pie y limitado a un **máximo de 3 actividades por día**.

---

### **Itinerario Personalizado: Buenos Aires (5 Días)**

#### **Día 1: Historia, Cultura Central y Gastronomía Tradicional (Monserrat y San Nicolás)**
* **Actividad 1 (Cultura e Historia):** Recorrido a pie por la histórica **Plaza de Mayo**, **Casa Rosada**, **Cabildo** y la **Catedral Metropolitana**.
* **Actividad 2 (Cultura):** Visita cultural a la zona de la **Avenida de Mayo** y el majestuoso **Teatro Colón**.
* **Actividad 3 (Gastronomía):** Experiencia gastronómica probando la tradicional **pizza porteña** o una pausa en un **Bar Notable** de San Nicolás.

#### **Día 2: Tango, Tradición e Identidad Porteña (San Telmo y La Boca